In [0]:
import os, sys
import pandas as pd
from datetime import timedelta

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, repo_root)

from src import config_loader as cl
from src import fhir_client as fc

In [0]:
dbutils.widgets.text("resource", "ALL", "Resource (ALL or e.g. Patient)")
dbutils.widgets.text("load_date", "2026-11-26", "Load date YYYY-MM-DD (blank = yesterday UTC)")
dbutils.widgets.text("max_pages", "", "Max pages override (blank = config)")
dbutils.widgets.dropdown("format", "json", ["json", "xml"], "Format")
dbutils.widgets.text("catalog", "", "Catalog override (optional)")

resource_param = dbutils.widgets.get("resource").strip()
load_date = dbutils.widgets.get("load_date").strip() or \
    (cl.utc_now() - timedelta(days=1)).strftime("%Y-%m-%d")
max_pages = int(dbutils.widgets.get("max_pages")) if dbutils.widgets.get("max_pages").strip() else None
fmt = dbutils.widgets.get("format")
catalog_override = dbutils.widgets.get("catalog").strip()

overrides = {"project": {"catalog": catalog_override}} if catalog_override else None
cfg = cl.load_config(os.path.join(repo_root, "config", "config.yaml"), overrides)

only = None if resource_param.upper() == "ALL" else [resource_param]
resources = cl.get_resources(cfg, only)
print(f"load_date={load_date} | format={fmt} | max_pages={max_pages or cfg['api']['max_pages_per_run']}")
print("Resources, in order:", [r["name"] for r in resources])

In [0]:
summaries = []
for r in resources:   # config order: Patient -> Encounter -> Observation -> Condition
    print(f"Fetching {r['name']} ...")
    s = fc.ingest_resource_to_raw(cfg, r["name"], load_date, max_pages=max_pages, fmt=fmt)
    summaries.append(s)
    print(f"  {s['status']}: {s['pages_fetched']} pages, {s['records_fetched']} records"
          f"{' (TRUNCATED by page cap)' if s['truncated'] else ''}")

display(pd.DataFrame(summaries)[["resource", "load_date", "status", "pages_fetched",
                                 "records_fetched", "truncated", "error", "raw_dir"]])

failed = [s for s in summaries if s["status"] != "SUCCESS"]
if failed:
    raise Exception(f"Ingestion failed for: {[(s['resource'], s['error']) for s in failed]}")

In [0]:
s = summaries[0]
files = sorted(os.listdir(s["raw_dir"]))
print(s["raw_dir"]); print(files[:5], "...")

manifest = pd.read_json(os.path.join(s["raw_dir"], "_manifest.jsonl"), lines=True)
display(manifest[["page_number", "entry_count", "bytes", "extracted_at", "api_url"]])

# peek at the first raw page (it should start with {"resourceType":"Bundle" ...)
with open(os.path.join(s["raw_dir"], "page_0001.json"), "r") as f:
    print(f.read(600))